<a href="https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one report date, in
fact_content_daily_performance (partitioned by month=YYYY-MM). Time window:
mid-panel month month=2026-03 for building/iterating. The _sample table is
the sealed final month (June 2026) — used only to test query mechanics,
never for label logic, since it's the natural outcome window of any
past-to-future label.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
path = f"{rel}/fact_content_daily_performance/month={MONTH}/*.parquet"

schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").df()
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields: clicks, impressions, position, ctr — all historical (trailing windows only)
Label field (proxy): whether next-period clicks rise vs. trailing baseline for the same content item
Context fields: client_id, content_id, report_date — identify the row, not predictive on their own
Excluded: GA4 engagement/session metrics — out of scope for this lane, would blur the contract.
Also excluded: _sample (June 2026) for anything label-related — it's the sealed test month.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

cols = schema['column_name'].tolist()
def find(*keywords):
    for c in cols:
        if all(k in c.lower() for k in keywords):
            return c
    return None

date_col, client_col, content_col = find("date"), find("client"), find("content")
clicks_col, impr_col, pos_col, ctr_col = find("click"), find("impress"), find("position"), find("ctr")
bool_cols = schema[schema['column_type'].str.contains('BOOL', case=False)]['column_name'].tolist()

print("date:", date_col, "client:", client_col, "content:", content_col)
print("clicks:", clicks_col, "impressions:", impr_col, "position:", pos_col, "ctr:", ctr_col)
print("boolean/availability columns:", bool_cols)


date: report_date client: client_hash_id content: content_hash_id
clicks: gsc_clicks impressions: gsc_impressions position: gsc_sum_position ctr: None
boolean/availability columns: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims, three queries: grain (one row = one date+client+content),
row count and date span of this slice, and availability (filtered with
IS TRUE, showing how many rows survive).

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# grain check
q1 = con.sql(f"""
    SELECT {date_col}, {client_col}, {content_col}, COUNT(*) AS c
    FROM read_parquet('{path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows:", len(q1), "(expect 0)")

# row count + date span
q2 = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN({date_col}) AS min_date, MAX({date_col}) AS max_date
    FROM read_parquet('{path}')
""").df()
print(q2)

# availability, IS TRUE
avail_col = bool_cols[0] if bool_cols else None
q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE {avail_col} IS TRUE) AS available_rows
    FROM read_parquet('{path}')
""").df()
print("Availability column used:", avail_col)
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: 0 (expect 0)
    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability column used: client_has_gsc
   total_rows  available_rows
0     9841378         9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is an unbalanced panel — dim_clients.gsc_data_start/ga4_data_start differ
per client, so month=2026-03 doesn't give every client equal history. Trailing-
window features will be thin or missing for newer clients. This data also can't
tell us why rankings moved (algorithm changes, competitor actions) — only that
they did.

In [18]:
q4 = con.sql(f"""
    SELECT gsc_data_start, COUNT(*) AS n_clients
    FROM read_parquet('{rel}/dim_clients.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()
q4

,gsc_data_start,n_clients
0,2025-01-27,2
1,2025-02-11,1
2,2025-03-11,1
3,2025-06-07,1
4,2025-06-18,1
5,2025-06-21,2
6,2025-06-29,1
7,2025-07-01,1
8,2025-07-06,1
9,2025-07-07,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.